In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from config_train import get_config
from torch.utils.data import DataLoader
from tsp import TSPDataset
from cvrp import CVRPDataset
import torch
import numpy as np
import time
import datetime
import tqdm
import os
import logging
import sys
from utils import read_instance_data
# import importlib
# importlib.reload(sys.modules['search_control'])
# importlib.reload(sys.modules['de'])
from search_control import solve_instance_set
from VAE_8 import VAE_8
# torch.set_float32_matmul_precision('high')
from torch.utils.tensorboard import SummaryWriter

def calculate_RC_loss(tour_logp):
    RC = - tour_logp.sum() / 2
    return RC


def calculate_KLD_loss(mean, log_var):
    KLD = -0.5 * torch.sum(1 + log_var - mean.pow(2) - log_var.exp()) / 2
    return KLD

In [3]:
def evaluate_network(config, model, validation_dataloader, epoch_idx):
    model.eval()
    loss_RC_values = []
    loss_KLD_values = []
    abs_Z_values = []
    for batch_id, batch in enumerate(validation_dataloader):
        instances, solutions_1, solutions_2 = batch

        with torch.no_grad():
            output, mean, log_var, Z, tour_idx, tour_logp = model(instances, solutions_1, solutions_2, config)
        loss_RC = calculate_RC_loss(tour_logp)
        loss_KLD = calculate_KLD_loss(mean, log_var)

        loss_RC_values.append(loss_RC.item())
        loss_KLD_values.append(loss_KLD.item())
        abs_Z = torch.abs(Z)  # Absolute coordinates of points in latent space (Z)
        abs_Z_values.append(abs_Z.cpu().numpy())

    abs_Z_values = np.array(abs_Z_values).flatten()

    # The bounds of the search space are defined as a percentile of the absolute latent variable coordinates
    new_bound = np.percentile(abs_Z_values, config.q_percentile).item()

    avgRC = np.mean(loss_RC_values)
    avgKL = np.mean(loss_KLD_values)
    writer.add_scalar("Loss/Reconstruction/Val", avgRC, epoch_idx)
    writer.add_scalar("Loss/KL-divergence/Val", avgKL, epoch_idx)
    writer.add_scalar("Loss/Combined/Val", avgRC + config.KLD_weight * avgKL, epoch_idx)

    return new_bound

In [4]:
VERSION = "0.4.0"
run_id = np.random.randint(10000, 99999)
now = datetime.datetime.now()

config = get_config(inJupyter=True)

if config.output_path == "":
    config.output_path = os.getcwd()
run_id = f"run_{now.day}.{now.month}_{now.hour}-{now.minute}-{now.second}_{run_id}"
config.output_path = os.path.join(config.output_path, "runs", run_id)
os.makedirs(os.path.join(config.output_path, "models"))

# why the fuck are configs written wrong into the tensorboard?
writer = SummaryWriter(log_dir=f"tensorboard_logdir/{run_id}")
writer.add_text('config', '\n'.join(map(lambda k_v: f"{k_v[0]}: {k_v[1]}", vars(config).items())))

logging.basicConfig(
    filename=os.path.join(config.output_path, "log_" + str(run_id) + ".txt"), filemode='w',
    level=logging.INFO, format='[%(levelname)s]%(message)s', force=True)
logging.info("Started Training Run")
logging.info("Call: {0}".format(''.join(sys.argv)))
logging.info("Version: {0}".format(VERSION))
logging.info("PARAMETERS:")
for arg in sorted(vars(config)):
    logging.info("{0}: {1}".format(arg, getattr(config, arg)))
logging.info("----------")
# training_data, validation_data = read_instance_data(config) # this takes 10 fucking seconds...

In [5]:
def train_step(instances, solutions_1, solutions_2):
    optimizer.zero_grad()
    output, mean, log_var, Z, tour_idx, tour_logp = model(instances, solutions_1, solutions_2, config)
    loss_RC = calculate_RC_loss(tour_logp)
    loss_KLD = calculate_KLD_loss(mean, log_var)
    loss = loss_RC + loss_KLD * config.KLD_weight
    assert not torch.isnan(loss)
    loss.backward()
    total_norm = 0.0
    for p in model.parameters():
        if p.grad is not None:
            param_norm = p.grad.data.norm(2)
            total_norm += param_norm.item() ** 2
    total_norm = total_norm ** 0.5
    print("grad_norm:", total_norm)

    # check for inf/nan in activations or params
    for name, p in model.named_parameters():
        if torch.isnan(p).any() or torch.isinf(p).any():
            print("bad param:", name)
    # torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
    optimizer.step()
    return loss_RC, loss_KLD

def train_epoch(epoch_idx):
    global training_instances, training_solutions
    model.train()
    # n = training_instances.shape[0]
    # perm = torch.randperm(n)
    # training_instances = training_instances[perm, :]
    # training_solutions = training_solutions[perm, :]
    # shift1 = torch.randint(high=config.problem_size, size=(1,))
    # s1 = torch.roll(training_solutions, int(shift1), 1)
    # shift2 = torch.randint(high=config.problem_size, size=(1,))
    # s2 = torch.roll(training_solutions, int(shift2), 1)

    # for batch_id in range((training_instances.shape[0] + config.batch_size - 1) // config.batch_size):
    #     sl = slice(batch_id * config.batch_size, (batch_id + 1) * config.batch_size)
    #     instances, solutions_1, solutions_2 = training_instances[sl], s1[sl], s2[sl]
    loss_RC_values = []
    loss_KLD_values = []
    for batch_id, batch in enumerate(training_dataloader):
        instances, solutions_1, solutions_2 = batch
        loss_RC, loss_KLD = train_step(instances, solutions_1, solutions_2)
        loss_RC_values.append(loss_RC.item())
        loss_KLD_values.append(loss_KLD.item())
        lr_scheduler.step()
    avgRC = np.mean(loss_RC_values)
    avgKL = np.mean(loss_KLD_values)
    writer.add_scalar("Loss/Reconstruction/Train", avgRC, epoch_idx)
    writer.add_scalar("Loss/KL-divergence/Train", avgKL, epoch_idx)
    writer.add_scalar("Loss/Combined/Train", avgRC + config.KLD_weight * avgKL, epoch_idx)
    writer.add_scalar("LR", lr_scheduler.get_last_lr()[0], epoch_idx)

In [6]:
def do(starting_epoch = 0):
    best_avg_gap = np.inf
    for epoch_idx in tqdm.trange(1, config.nb_epochs + 1):
        epoch_idx += starting_epoch
        train_epoch(epoch_idx)
        if epoch_idx > 100 and epoch_idx % 50 == 0:
            new_bound = evaluate_network(config, model, validation_dataloader, epoch_idx)

            config.search_space_bound = new_bound
            writer.add_scalar("Search Space Bounds", new_bound, epoch_idx)

            avg_gap, avg_runtime, _ = solve_instance_set(model, config,
                                                         validation_data[0][: config.search_validation_size]
                                                         , validation_data[1][:config.search_validation_size])

            # If the average gap is improved, save the model
            if avg_gap < best_avg_gap:
                best_avg_gap = avg_gap
                model_data = {
                    'parameters': model.state_dict(),
                    'code_version': VERSION,
                    'problem': config.problem,
                    'problem_size': config.problem_size,
                    'Z_bound': new_bound,
                    'avg_gap': avg_gap,
                    'training_epochs': epoch_idx,
                    'model': "VAE_final"
                }

                torch.save(model_data, os.path.join(config.output_path, "models",
                                                    "model_{0}.pt".format(run_id, epoch_idx)))

            writer.add_scalar("Gap/Val", avg_gap * 100, epoch_idx)
            writer.add_scalar("Search Time/Val", avg_runtime, epoch_idx)
    return best_avg_gap

In [7]:
model = VAE_8(config).to(config.device)
optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)
training_data = torch.load('training_data.torch', weights_only=False)

In [8]:
validation_data = torch.load('validation_data.torch', weights_only=False)
training_dataset = TSPDataset(config.epoch_size, config.problem_size, config, training_data)
training_dataloader = DataLoader(training_dataset, batch_size=config.batch_size, num_workers=0, shuffle=True)
validation_dataset = TSPDataset(config.network_validation_size, config.problem_size, config, validation_data)
validation_dataloader = DataLoader(validation_dataset, batch_size=config.batch_size, num_workers=0, shuffle=True)
lr_scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, config.lr, epochs=config.nb_epochs, steps_per_epoch=len(training_dataloader))

In [ ]:
avg_gap = do()
writer.add_hparams(config, {"hparam/avg_gap": avg_gap * 100})

  0%|                                                   | 0/300 [00:00<?, ?it/s]

grad_norm: 39380.7452962347
grad_norm: 40629.60600911423
grad_norm: 45211.430718501375
grad_norm: 46084.505015261544
grad_norm: 47535.94183006431
grad_norm: 37592.31740287364
grad_norm: 49898.16061600409
grad_norm: 31518.88966092863
grad_norm: 30559.540641132447
grad_norm: 41494.63140586599
grad_norm: 38514.26278749617
grad_norm: 30527.39714896758
grad_norm: 38658.43439647309
grad_norm: 28274.60078351478
grad_norm: 30551.069574849436
grad_norm: 31730.83025621124
grad_norm: 25223.324166058403
grad_norm: 38405.150181305326
grad_norm: 21310.345623873378
grad_norm: 25405.41461130355
grad_norm: 19388.2287832331
grad_norm: 22287.700430339155
grad_norm: 21603.061652046068
grad_norm: 30915.507970343726
grad_norm: 24098.894524511994
grad_norm: 24952.913985096846
grad_norm: 28936.597585052095
grad_norm: 14900.14047778845
grad_norm: 17823.208934570277
grad_norm: 12931.938723249272
grad_norm: 11723.825584348588
grad_norm: 13519.82754546347
grad_norm: 17854.82034143536
grad_norm: 17261.591201560208

grad_norm: 1085.6613555719066
grad_norm: 5508.280818696188
grad_norm: 3881.4811380382775
grad_norm: 7459.653584536982
grad_norm: 3462.350120442297
grad_norm: 6681.929814497533
grad_norm: 3509.976768073244
grad_norm: 5461.061039379377
grad_norm: 8126.854757650564
grad_norm: 7323.163104095382
grad_norm: 1570.4440297888912
grad_norm: 5360.967758993018
grad_norm: 4242.102515376418
grad_norm: 10462.572672746399
grad_norm: 11100.45497099231
grad_norm: 13253.742936748788
grad_norm: 4606.691217319532
grad_norm: 7828.2241717333445
grad_norm: 2385.4098131807154
grad_norm: 13521.477860835726
grad_norm: 18492.979634336723
grad_norm: 5992.638906775581
grad_norm: 7221.265423004858
grad_norm: 10131.7220668091
grad_norm: 1580.490455586313
grad_norm: 2534.0196113460092
grad_norm: 2303.913654418058
grad_norm: 7766.063996806111
grad_norm: 12100.667379077606
grad_norm: 18276.17974111731
grad_norm: 2721.6170814381626
grad_norm: 9160.676181820263
grad_norm: 5722.9427240014165
grad_norm: 16889.99592235838
gr

  0%|▏                                        | 1/300 [00:43<3:36:30, 43.45s/it]

grad_norm: 11287.193431165151
grad_norm: 4847.440813975414
grad_norm: 8481.828483594427
grad_norm: 12849.854780082938
grad_norm: 2444.5095338593205
grad_norm: 3523.9313009395887
grad_norm: 9200.71068481266
grad_norm: 7243.881810154304
grad_norm: 3790.656239537382
grad_norm: 14468.667133762025
grad_norm: 8657.839289029283
grad_norm: 5191.773304191968
grad_norm: 12576.129084650936
grad_norm: 7253.999516690634
grad_norm: 12546.837698008745
grad_norm: 5617.582419043764
grad_norm: 5318.255971029485
grad_norm: 5367.011528173794
grad_norm: 8713.278553443506
grad_norm: 9691.297276457899
grad_norm: 4590.127002906198
grad_norm: 10196.85200687092
grad_norm: 11626.103846746559
grad_norm: 6960.557896354038
grad_norm: 4453.6465612392085
grad_norm: 4702.061268286269
grad_norm: 7817.954669941643
grad_norm: 2090.731764403402
grad_norm: 2465.3107421366444
grad_norm: 1718.5224297265745
grad_norm: 8109.415577979798
grad_norm: 5102.6003913808345
grad_norm: 17175.87335809607
grad_norm: 14623.30277619837
gra

...

 36%|█████████████▎                       | 108/300 [1:18:00<2:18:48, 43.38s/it]

grad_norm: 45308.943872199925
grad_norm: 41537.85659095536
grad_norm: 72984.80624677434
grad_norm: 69543.14378716443
grad_norm: 43734.909725786994
grad_norm: 49384.907009668364
grad_norm: 58359.100686654085
grad_norm: 49725.65738501108
grad_norm: 47672.50357768191
grad_norm: 49562.22748116403
grad_norm: 54126.66127023463
grad_norm: 52257.123571249576
grad_norm: 66184.02219428848
grad_norm: 47645.51201894479
grad_norm: 51348.02181497955
grad_norm: 48594.13540847983
grad_norm: 51265.27441716854
grad_norm: 45240.71191862499
grad_norm: 42441.55615677899
grad_norm: 57933.8998277627
grad_norm: 59246.448237554694
grad_norm: 47753.83251781463
grad_norm: 42155.82349051216
grad_norm: 50405.13856041906
grad_norm: 46509.275758684555
grad_norm: 46745.54141572493
grad_norm: 45164.43947645879
grad_norm: 57036.137593338375
grad_norm: 59207.411011409386
grad_norm: 50745.98264837679
grad_norm: 53523.403815908154
grad_norm: 53828.165325301634
grad_norm: 51755.49105359414
grad_norm: 52862.532820450186
gra

grad_norm: 41203.07933254189
grad_norm: 48373.71011499213
grad_norm: 52007.35253300452
grad_norm: 51107.59466242091
grad_norm: 50677.6052737421
grad_norm: 48793.00098981694
grad_norm: 80729.1864819172
grad_norm: 54951.303366313005
grad_norm: 57239.34351104023
grad_norm: 56379.15156361059
grad_norm: 62073.85561838731
grad_norm: 46984.937961038064
grad_norm: 57663.18116656303
grad_norm: 53673.336809687615
grad_norm: 45315.5938659587
grad_norm: 64493.057035101345
grad_norm: 49056.06450395098
grad_norm: 56540.438146863504
grad_norm: 49108.237528945974
grad_norm: 54715.78686451228
grad_norm: 44048.08525945431
grad_norm: 44714.39330741767
grad_norm: 48617.47919286339
grad_norm: 46543.913327661
grad_norm: 61333.58013792652
grad_norm: 47031.98714907883
grad_norm: 58487.29711742086
grad_norm: 51530.7043064577
grad_norm: 43783.801111256056
grad_norm: 45918.56455285725
grad_norm: 47480.591189063125
grad_norm: 49756.01349000454
grad_norm: 47479.46030490514
grad_norm: 60672.87526287323
grad_norm: 5

 36%|█████████████▍                       | 109/300 [1:18:43<2:17:59, 43.35s/it]

grad_norm: 51040.48045829798
grad_norm: 69224.51992740399
grad_norm: 43254.30966994173
grad_norm: 55038.101105531576
grad_norm: 69067.59835010752
grad_norm: 53208.54815922424
grad_norm: 53512.05857408672
grad_norm: 47595.462685716804
grad_norm: 43261.83598352765
grad_norm: 53182.5791045783
grad_norm: 61535.3594717752
grad_norm: 42415.01104899851
grad_norm: 51541.49372202121
grad_norm: 58412.18981676738
grad_norm: 51237.05655038661
grad_norm: 47819.381659460945
grad_norm: 59491.152790401204
grad_norm: 45420.069496689284
grad_norm: 45327.478400040694
grad_norm: 53445.997482535575
grad_norm: 45303.06724937916
grad_norm: 39579.88438359601
grad_norm: 45317.36067939012
grad_norm: 42225.05340215413
grad_norm: 45594.414738594416
grad_norm: 42073.67623735078
grad_norm: 39093.109509009235
grad_norm: 44778.37317388434
grad_norm: 49993.32249489287
grad_norm: 41450.48156987169
grad_norm: 49270.1341341122
grad_norm: 49012.05287361059
grad_norm: 50967.27769084605
grad_norm: 49834.49495271641
grad_nor